# Análise estatística dos resultados do pentest (Capítulo 4)

Notebook da análise estatística do TCC *Análise de Vulnerabilidades em Aplicações Web Desenvolvidas com IA Generativa (vibe coding) e em Plataformas Low-Code: um estudo baseado no OWASP Top 10*.

**Entrada:** os arquivos gerados pelo `zap/consolidar_zap.py` na pasta `relatorios_zap/consolidado/`. São obrigatórios `achados.csv` e `resumo_apps.csv`; os demais não são usados, porque toda a análise é recalculada aqui a partir desses dois.

**Saída:** pasta `resultados_cap4/` com as tabelas (`tabelas_cap4.xlsx` e um CSV por tabela) e as figuras em PNG de 300 dpi, prontas para o texto.

**Como usar no Google Colab:**
1. Rode a rodada final no seu computador: `python zap/consolidar_zap.py relatorios_zap --excluir-cabecalhos --excluir-pendentes`.
2. Abra este notebook no Colab, mantenha `ORIGEM_DADOS = "upload"` na célula de configuração e execute *Ambiente de execução → Executar tudo*.
3. Quando aparecer o botão de upload, envie `achados.csv` e `resumo_apps.csv` (ou um `.zip` da pasta `consolidado`).
4. No final, o Colab baixa um `.zip` com as tabelas e as figuras.

Também roda em Jupyter local (`ORIGEM_DADOS = "local"`) ou lendo do Google Drive (`ORIGEM_DADOS = "drive"`).

**Unidade de análise:** a aplicação. Um desfecho (por exemplo, "tem achado em A05") vale 1 se a aplicação tiver ao menos um alerta incluído naquela categoria. A comparação principal é vibe coding (exposição) contra low-code (referência), estratificada pelo domínio funcional, conforme o plano do `README.md`.

In [ ]:
# === CONFIGURAÇÃO ===
ORIGEM_DADOS = "local"   # "upload" (Colab), "drive" (Google Drive) ou "local" (Jupyter)
PASTA_DRIVE = "/content/drive/MyDrive/pesquisa-pgt/relatorios_zap/consolidado"
PASTA_LOCAL = "../relatorios_zap/consolidado"
PASTA_SAIDA = "resultados_cap4"
ALFA = 0.05              # nível de significância (README)
ARQUIVO_SEGREDOS = None  # opcional (H2): caminho de um CSV com colunas app;segredo (0/1)

## Instalação das bibliotecas

In [ ]:
%pip install -q "pandas>=2.0" "numpy>=1.24" "scipy>=1.10" "statsmodels>=0.14" "matplotlib>=3.7" "openpyxl>=3.1"


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, io, glob, math, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import StratifiedTable
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

try:
    import google.colab  # noqa: F401
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

# ---- Rótulos (README) ----
CATEGORIAS = ["A01", "A02", "A03", "A04", "A05", "A06", "A07", "A08", "A09", "A10"]
NOME_CAT = {
    "A01": "Broken Access Control", "A02": "Security Misconfiguration",
    "A03": "Software Supply Chain Failures", "A04": "Cryptographic Failures",
    "A05": "Injection", "A06": "Insecure Design", "A07": "Authentication Failures",
    "A08": "Software or Data Integrity Failures",
    "A09": "Security Logging and Alerting Failures",
    "A10": "Mishandling of Exceptional Conditions"}
ROTULO_FONTE = {
    "claude_sonnet": "Claude Sonnet 4.5", "deepseek": "DeepSeek V4.1 Flash",
    "gemini_flash": "Gemini 3.8 Flash", "gpt5": "GPT-5.5", "kimi": "Kimi K3",
    "qwen": "Qwen3.8 27B", "bolt": "Bolt.new", "lovable": "Lovable"}
ROTULO_GRUPO = {"vibe": "Vibe coding", "lowcode": "Low-code"}
ORDEM_FONTES = ["claude_sonnet", "deepseek", "gemini_flash", "gpt5", "kimi", "qwen",
                "bolt", "lovable"]
GRUPO_EXPOSTO, GRUPO_REFERENCIA = "vibe", "lowcode"

# ---- Estilo dos gráficos (paleta validada: 2 cores categóricas + rampa azul) ----
COR_GRUPO = {"vibe": "#2a78d6", "lowcode": "#eb6834"}
COR_RISCO = {"Baixo": "#86b6ef", "Medio": "#2a78d6", "Alto": "#104281"}
TINTA, TINTA_2, GRADE = "#0b0b0b", "#52514e", "#e4e3df"
RAMPA_AZUL = LinearSegmentedColormap.from_list(
    "azul", ["#f4f8fd", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf",
             "#184f95", "#0d366b"])
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.edgecolor": TINTA_2, "axes.labelcolor": TINTA, "text.color": TINTA,
    "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "legend.frameon": False})

os.makedirs(os.path.join(PASTA_SAIDA, "figuras"), exist_ok=True)
TABELAS = {}   # tabelas exportadas no final


def num(x, casas=1):
    # Formata número com vírgula decimal (padrão do texto em português).
    if x is None or (isinstance(x, float) and (math.isnan(x))):
        return "NA"
    if isinstance(x, float) and math.isinf(x):
        return "infinito"
    return f"{x:.{casas}f}".replace(".", ",")


def salvar_figura(fig, nome):
    caminho = os.path.join(PASTA_SAIDA, "figuras", nome + ".png")
    fig.savefig(caminho)
    print("Figura salva:", caminho)

## Leitura dos dados

In [ ]:
def pasta_de_entrada():
    if ORIGEM_DADOS == "local":
        return PASTA_LOCAL
    if ORIGEM_DADOS == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        return PASTA_DRIVE
    if ORIGEM_DADOS == "upload":
        from google.colab import files
        destino = "/content/consolidado"
        os.makedirs(destino, exist_ok=True)
        enviados = files.upload()
        for nome, conteudo in enviados.items():
            if nome.lower().endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
                    for membro in z.namelist():
                        if membro.lower().endswith(".csv"):
                            with open(os.path.join(destino, os.path.basename(membro)), "wb") as f:
                                f.write(z.read(membro))
            else:
                with open(os.path.join(destino, nome), "wb") as f:
                    f.write(conteudo)
        return destino
    raise ValueError("ORIGEM_DADOS deve ser 'upload', 'drive' ou 'local'.")


def ler_csv(caminho):
    # Aceita o formato padrão do consolidar_zap.py (; e vírgula decimal) e o --csv-padrao.
    df = pd.read_csv(caminho, sep=";", decimal=",", encoding="utf-8-sig", dtype=str)
    if df.shape[1] == 1:
        df = pd.read_csv(caminho, sep=",", decimal=".", encoding="utf-8-sig", dtype=str)
    return df


PASTA = pasta_de_entrada()
achados = ler_csv(os.path.join(PASTA, "achados.csv"))
resumo_apps = ler_csv(os.path.join(PASTA, "resumo_apps.csv"))

for col in ["risco_cod", "confianca_cod", "incluido_analise", "n_instancias"]:
    achados[col] = pd.to_numeric(achados[col], errors="coerce").fillna(0).astype(int)
for col in ["endpoints", "pct_respostas_5xx"]:
    resumo_apps[col] = pd.to_numeric(resumo_apps[col], errors="coerce")
achados["owasp_2025"] = achados["owasp_2025"].fillna("")

# Universo de aplicações testadas (inclui as que não tiveram nenhum achado).
apps = resumo_apps[resumo_apps["relatorio"] == "sim"][["grupo", "fonte", "dominio", "app"]].copy()
apps = apps.reset_index(drop=True)
print(f"Aplicações com relatório: {len(apps)}")
print(apps.groupby("grupo").size().rename("n_apps").to_string())

pendentes = achados[achados["validacao"] == "pendente"]
if len(pendentes):
    print(f"\nATENÇÃO: {len(pendentes)} alertas ainda aguardam validação manual em validacao.csv."
          " Os resultados abaixo são provisórios até a rodada final.")
if (apps["grupo"] == GRUPO_REFERENCIA).sum() == 0:
    print("\nATENÇÃO: ainda não há aplicações low-code; a comparação entre grupos fica para depois.")

In [ ]:
# Dois cenários de contagem, ambos sobre a rodada final (sem cabeçalhos e sem pendentes).
# principal: tudo o que o consolidar_zap.py marcou em "incluido_analise".
# conservador: dentro do principal, só alertas validados como VP ou, quando a
#   validação não é exigida, com confiança Média ou superior. Na prática, retira
#   os alertas de confiança Baixa, como o 40014-2 ("XSS persistente na resposta JSON").
def filtro_principal(df):
    return df["incluido_analise"] == 1

def filtro_conservador(df):
    return (df["incluido_analise"] == 1) & (
        (df["validacao"] == "VP") |
        ((df["validacao"] == "nao_requerida") & (df["confianca_cod"] >= 2)))

CENARIOS = {
    "principal": filtro_principal,
    "conservador": filtro_conservador,
}
ROTULO_CENARIO = {"principal": "Cenário principal", "conservador": "Cenário conservador"}


def presenca(filtro):
    # Uma linha por aplicação, com 0/1 em cada desfecho.
    inc = achados[filtro(achados)]
    tab = apps.copy()
    for cat in CATEGORIAS:
        tab[cat] = tab["app"].isin(inc.loc[inc["owasp_2025"] == cat, "app"]).astype(int)
    tab["qualquer_achado"] = tab["app"].isin(inc["app"]).astype(int)
    tab["medio_ou_mais"] = tab["app"].isin(inc.loc[inc["risco_cod"] >= 2, "app"]).astype(int)
    tab["alto"] = tab["app"].isin(inc.loc[inc["risco_cod"] == 3, "app"]).astype(int)
    contagem = inc.groupby("app").size()
    tab["n_achados"] = tab["app"].map(contagem).fillna(0).astype(int)
    for cod, nome in [(3, "n_alto"), (2, "n_medio"), (1, "n_baixo")]:
        tab[nome] = tab["app"].map(inc[inc["risco_cod"] == cod].groupby("app").size()).fillna(0).astype(int)
    return tab, inc


PRES, INC = presenca(filtro_principal)
PRES_CONS, INC_CONS = presenca(filtro_conservador)
DESFECHOS = CATEGORIAS + ["qualquer_achado", "medio_ou_mais", "alto"]
print("Achados incluídos no cenário principal:  ", len(INC))
print("Achados incluídos no cenário conservador:", len(INC_CONS))
retirados = INC[~INC.index.isin(INC_CONS.index)]
if len(retirados):
    print("\nAlertas que saem no cenário conservador (n de aplicações):")
    display(retirados.groupby(["alertRef", "alerta", "risco", "confianca"])["app"].nunique()
            .sort_values(ascending=False).rename("n_apps").reset_index())

## 4.1 Descrição da amostra

Quantas aplicações foram testadas em cada fonte, quantas não funcionaram (sem relatório: erro grave impediu o teste) e a cobertura de cada varredura (endpoints encontrados pelo ZAP e proporção de respostas 5xx).

In [ ]:
previstas = resumo_apps.groupby(["grupo", "fonte"]).size().rename("previstas")
testadas = resumo_apps[resumo_apps["relatorio"] == "sim"].groupby(["grupo", "fonte"]).agg(
    testadas=("app", "count"),
    endpoints_mediana=("endpoints", "median"),
    endpoints_min=("endpoints", "min"),
    endpoints_max=("endpoints", "max"),
    pct_5xx_mediana=("pct_respostas_5xx", "median"))
amostra = pd.concat([previstas, testadas], axis=1).fillna({"testadas": 0}).reset_index()
amostra["testadas"] = amostra["testadas"].astype(int)
amostra["fonte"] = pd.Categorical(amostra["fonte"], ORDEM_FONTES, ordered=True)
amostra = amostra.sort_values(["grupo", "fonte"], ascending=[False, True])
amostra["fonte"] = amostra["fonte"].map(ROTULO_FONTE)
amostra["grupo"] = amostra["grupo"].map(ROTULO_GRUPO)
TABELAS["4.1_amostra"] = amostra
display(amostra)

ausentes = resumo_apps.loc[resumo_apps["relatorio"] != "sim", "app"].tolist()
print(f"Aplicações previstas sem relatório ({len(ausentes)}):")
print(", ".join(ausentes) if ausentes else "nenhuma")

## 4.2 Distribuição dos achados por aplicação

Contagem de tipos de alerta distintos por aplicação (cenário principal), por fonte e por grupo.

In [ ]:
def descritiva(df, chaves):
    g = df.groupby(chaves)
    out = g.agg(n_apps=("app", "count"), total=("n_achados", "sum"),
                media=("n_achados", "mean"), mediana=("n_achados", "median"),
                desvio_padrao=("n_achados", "std"), minimo=("n_achados", "min"),
                maximo=("n_achados", "max"),
                pct_apps_alto=("alto", "mean"), pct_apps_medio_ou_mais=("medio_ou_mais", "mean"))
    out[["pct_apps_alto", "pct_apps_medio_ou_mais"]] *= 100
    return out.round(2).reset_index()

desc_fonte = descritiva(PRES, ["grupo", "fonte"])
desc_fonte["fonte"] = pd.Categorical(desc_fonte["fonte"], ORDEM_FONTES, ordered=True)
desc_fonte = desc_fonte.sort_values(["grupo", "fonte"], ascending=[False, True])
desc_grupo = descritiva(PRES, ["grupo"]).assign(fonte="(total do grupo)")
desc = pd.concat([desc_fonte, desc_grupo], ignore_index=True)
desc["fonte"] = desc["fonte"].astype(str).map(lambda f: ROTULO_FONTE.get(f, f))
desc["grupo"] = desc["grupo"].map(ROTULO_GRUPO)
TABELAS["4.2_descritiva"] = desc
display(desc)

In [ ]:
# Figura 1: achados por aplicação, um ponto por app, mediana marcada.
fontes_presentes = [f for f in ORDEM_FONTES if f in set(PRES["fonte"])]
fig, ax = plt.subplots(figsize=(7, 0.45 * len(fontes_presentes) + 1.2))
rng = np.random.default_rng(42)
for i, f in enumerate(fontes_presentes):
    sub = PRES[PRES["fonte"] == f]
    grupo = sub["grupo"].iloc[0]
    y = i + rng.uniform(-0.15, 0.15, len(sub))
    ax.scatter(sub["n_achados"], y, s=36, color=COR_GRUPO[grupo], edgecolor="white",
               linewidth=1, zorder=3)
    ax.plot([sub["n_achados"].median()] * 2, [i - 0.3, i + 0.3], color=TINTA, lw=2, zorder=4)
ax.set_yticks(range(len(fontes_presentes)))
ax.set_yticklabels([ROTULO_FONTE[f] for f in fontes_presentes])
ax.invert_yaxis()
ax.set_xlabel("Tipos de alerta distintos por aplicação")
ax.set_title("Achados por aplicação, por fonte (traço preto = mediana)")
ax.grid(axis="y", visible=False)
for g in ["vibe", "lowcode"]:
    if g in set(PRES["grupo"]):
        ax.scatter([], [], s=36, color=COR_GRUPO[g], label=ROTULO_GRUPO[g])
ax.legend(loc="lower right")
salvar_figura(fig, "fig1_achados_por_app")
plt.show()

In [ ]:
# Figura 2: média de alertas por aplicação, decomposta por nível de risco.
medias = PRES.groupby("fonte")[["n_baixo", "n_medio", "n_alto"]].mean().reindex(fontes_presentes)
fig, ax = plt.subplots(figsize=(7, 0.45 * len(fontes_presentes) + 1.2))
esquerda = np.zeros(len(medias))
for col, rot in [("n_alto", "Alto"), ("n_medio", "Medio"), ("n_baixo", "Baixo")]:
    ax.barh(range(len(medias)), medias[col], left=esquerda, height=0.6,
            color=COR_RISCO[rot], edgecolor="white", linewidth=2,
            label=rot.replace("Medio", "Médio"))
    esquerda += medias[col].values
for i, total in enumerate(esquerda):
    ax.text(total + 0.1, i, num(total), va="center", color=TINTA_2, fontsize=9)
ax.set_yticks(range(len(medias)))
ax.set_yticklabels([ROTULO_FONTE[f] for f in medias.index])
ax.invert_yaxis()
ax.set_xlabel("Média de tipos de alerta por aplicação")
ax.set_title("Média de alertas por aplicação, por nível de risco")
ax.grid(axis="y", visible=False)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.18))
salvar_figura(fig, "fig2_risco_por_fonte")
plt.show()

## 4.3 Prevalência por categoria do OWASP Top 10:2025

Proporção de aplicações com ao menos um achado em cada categoria, com intervalo de confiança de 95% pelo método de Wilson.

In [ ]:
def prevalencias(pres, chave):
    linhas = []
    for valor, sub in pres.groupby(chave):
        n = len(sub)
        for d in DESFECHOS:
            x = int(sub[d].sum())
            lo, hi = proportion_confint(x, n, alpha=ALFA, method="wilson")
            linhas.append({chave: valor, "desfecho": d, "n": n, "casos": x,
                           "pct": 100 * x / n, "ic95_inf": 100 * lo, "ic95_sup": 100 * hi})
    return pd.DataFrame(linhas)

prev_grupo = prevalencias(PRES, "grupo")
prev_fonte = prevalencias(PRES, "fonte")

tab_prev = prev_grupo.pivot(index="desfecho", columns="grupo", values=["casos", "n", "pct", "ic95_inf", "ic95_sup"])
tab_prev = tab_prev.reindex(DESFECHOS)
tab_prev.columns = [f"{m}_{g}" for m, g in tab_prev.columns]
tab_prev.insert(0, "categoria", [NOME_CAT.get(d, "") for d in tab_prev.index])
tab_prev = tab_prev.round(1).reset_index()
TABELAS["4.3_prevalencia_grupo"] = tab_prev
display(tab_prev)

mapa = prev_fonte[prev_fonte["desfecho"].isin(CATEGORIAS)].pivot(index="fonte", columns="desfecho", values="pct")
mapa = mapa.reindex(fontes_presentes)[CATEGORIAS]
tab_mapa = mapa.round(1).rename(index=ROTULO_FONTE).reset_index()
TABELAS["4.3_prevalencia_fonte"] = tab_mapa
display(tab_mapa)

In [ ]:
# Figura 3: prevalência por categoria, comparando os grupos (barras + IC 95%).
grupos = [g for g in ["vibe", "lowcode"] if g in set(PRES["grupo"])]
fig, ax = plt.subplots(figsize=(7.5, 5.2))
altura = 0.8 / len(grupos)
for j, g in enumerate(grupos):
    sub = prev_grupo[(prev_grupo["grupo"] == g)].set_index("desfecho").reindex(CATEGORIAS)
    y = np.arange(len(CATEGORIAS)) + (j - (len(grupos) - 1) / 2) * altura
    ax.barh(y, sub["pct"], height=altura * 0.9, color=COR_GRUPO[g], edgecolor="white",
            linewidth=1, label=f"{ROTULO_GRUPO[g]} (n = {int(sub['n'].iloc[0])})")
    ax.errorbar(sub["pct"], y, xerr=[sub["pct"] - sub["ic95_inf"], sub["ic95_sup"] - sub["pct"]],
                fmt="none", ecolor=TINTA_2, elinewidth=1, capsize=2)
ax.set_yticks(range(len(CATEGORIAS)))
ax.set_yticklabels([f"{c} {NOME_CAT[c]}" for c in CATEGORIAS])
ax.invert_yaxis()
ax.set_xlim(0, 105)
ax.set_xlabel("% de aplicações com ao menos um achado (IC 95% de Wilson)")
ax.set_title("Prevalência de achados por categoria do OWASP Top 10:2025", pad=28)
ax.grid(axis="y", visible=False)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
salvar_figura(fig, "fig3_prevalencia_owasp_grupos")
plt.show()

In [ ]:
# Figura 4: mapa de calor fonte x categoria (% de aplicações com achado).
fig, ax = plt.subplots(figsize=(8.5, 0.45 * len(fontes_presentes) + 1.6))
im = ax.imshow(mapa.values, cmap=RAMPA_AZUL, vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(CATEGORIAS)))
ax.set_xticklabels(CATEGORIAS)
ax.set_yticks(range(len(mapa.index)))
ax.set_yticklabels([ROTULO_FONTE[f] for f in mapa.index])
ax.grid(False)
for i in range(mapa.shape[0]):
    for j in range(mapa.shape[1]):
        v = mapa.values[i, j]
        ax.text(j, i, num(v, 0), ha="center", va="center", fontsize=8,
                color="white" if v >= 55 else TINTA)
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.set_label("% de aplicações com achado")
cb.outline.set_visible(False)
ax.set_title("Prevalência por categoria do OWASP Top 10:2025, por fonte")
salvar_figura(fig, "fig4_mapa_owasp_fontes")
plt.show()

## 4.4 Alertas mais frequentes

Tipos de alerta do ZAP ordenados pelo número de aplicações em que aparecem (cenário principal).

In [ ]:
n_grupo = PRES.groupby("grupo").size()
nome_ref = INC.groupby("alertRef")["alerta"].agg(lambda s: s.value_counts().index[0])
freq = INC.groupby("alertRef").agg(
    risco=("risco", lambda s: s.iloc[0]), cwe=("cwe", "first"),
    owasp_2025=("owasp_2025", "first"), tipo=("tipo", "first"), n_apps=("app", "nunique"))
freq.insert(0, "alerta", nome_ref)
for g in ["vibe", "lowcode"]:
    n_apps_g = INC[INC["grupo"] == g].groupby("alertRef")["app"].nunique()
    freq[f"pct_apps_{g}"] = (100 * n_apps_g / n_grupo[g]).round(1) if g in n_grupo else np.nan
freq["pct_apps_total"] = (100 * freq["n_apps"] / len(PRES)).round(1)
freq["risco"] = freq["risco"].replace({"Medio": "Médio"})
freq = freq.sort_values(["n_apps", "risco"], ascending=[False, True]).reset_index()
TABELAS["4.4_alertas_frequentes"] = freq
display(freq.head(20))

## 4.5 Testes de hipóteses: vibe coding × low-code

Para cada desfecho, a tabela 2×2 compara as aplicações de vibe coding (exposição) com as de low-code (referência):

* **Odds ratio bruto** com IC 95% de Woolf (correção de Haldane-Anscombe, +0,5 em todas as células, quando há célula zero) e **p do teste exato de Fisher** bilateral.
* **Odds ratio de Mantel-Haenszel** estratificado por domínio funcional, com IC 95% de Robins-Breslow-Greenland, e **p do teste de Cochran-Mantel-Haenszel**, com correção de continuidade de 0,5 aplicada apenas quando |O − E| ≥ 0,5 (mesma regra do `consolidar_zap.py`).
* OR > 1 indica maior chance do achado em vibe coding.

Os testes são calculados nos dois cenários: **principal** (todos os alertas incluídos na rodada final) e **conservador** (só alertas validados como VP ou com confiança Média ou superior; retira, por exemplo, o 40014-2). A **H1** é testada pelos desfechos A05 (Injection) e A07 (Authentication Failures): a hipótese prevê OR > 1 nesses dois. Se a conclusão mudar entre os cenários, o resultado depende dos alertas de confiança Baixa e deve ser apresentado assim no texto. As demais categorias são exploratórias; para elas há também o p ajustado por Holm sobre as dez categorias, a ser usado ou não conforme o critério definido na metodologia.

In [ ]:
def or_bruto(a, b, c, d):
    obs = ""
    if 0 in (a, b, c, d):
        a, b, c, d = a + .5, b + .5, c + .5, d + .5
        obs = "correção +0,5"
    orr = (a * d) / (b * c)
    ep = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    z = 1.959963984540054
    return orr, math.exp(math.log(orr) - z * ep), math.exp(math.log(orr) + z * ep), obs


def mantel_haenszel(pres, d):
    tabs = []
    for _, sub in pres.groupby("dominio"):
        e = sub.loc[sub["grupo"] == GRUPO_EXPOSTO, d]
        r = sub.loc[sub["grupo"] == GRUPO_REFERENCIA, d]
        if len(e) == 0 or len(r) == 0:
            continue
        tabs.append(np.array([[e.sum(), len(e) - e.sum()], [r.sum(), len(r) - r.sum()]], dtype=float))
    if not tabs:
        return np.nan, np.nan, np.nan, np.nan, 0
    R = sum(t[0, 0] * t[1, 1] / t.sum() for t in tabs)
    S = sum(t[0, 1] * t[1, 0] / t.sum() for t in tabs)
    st = StratifiedTable(tabs)
    if R > 0 and S > 0:
        orr = st.oddsratio_pooled
        lo, hi = st.oddsratio_pooled_confint(alpha=ALFA)
    else:
        orr = 0.0 if (R == 0 and S > 0) else (math.inf if S == 0 and R > 0 else np.nan)
        lo = hi = np.nan
    soma_a = sum(t[0, 0] for t in tabs)
    soma_e = sum(t[0].sum() * t[:, 0].sum() / t.sum() for t in tabs)
    correcao = abs(soma_a - soma_e) >= 0.5
    try:
        p = st.test_null_odds(correction=correcao).pvalue
    except Exception:
        p = np.nan
    return orr, lo, hi, p, len(tabs)


def testar(pres, desfechos=DESFECHOS):
    exp_ = pres[pres["grupo"] == GRUPO_EXPOSTO]
    ref_ = pres[pres["grupo"] == GRUPO_REFERENCIA]
    if len(exp_) == 0 or len(ref_) == 0:
        return None
    linhas = []
    for d in desfechos:
        a, c = int(exp_[d].sum()), int(ref_[d].sum())
        b, dd = len(exp_) - a, len(ref_) - c
        orr, lo, hi, obs = or_bruto(a, b, c, dd)
        p_f = fisher_exact([[a, b], [c, dd]], alternative="two-sided").pvalue
        mh, mh_lo, mh_hi, p_mh, n_est = mantel_haenszel(pres, d)
        linhas.append({
            "desfecho": d, "categoria": NOME_CAT.get(d, ""),
            "hipotese": "H1" if d in ("A05", "A07") else "exploratória",
            "casos_vibe": a, "n_vibe": len(exp_), "pct_vibe": 100 * a / len(exp_),
            "casos_lowcode": c, "n_lowcode": len(ref_), "pct_lowcode": 100 * c / len(ref_),
            "or_bruto": orr, "or_ic95_inf": lo, "or_ic95_sup": hi, "p_fisher": p_f,
            "or_mh": mh, "or_mh_ic95_inf": mh_lo, "or_mh_ic95_sup": mh_hi, "p_cmh": p_mh,
            "estratos": n_est, "obs": obs})
    res = pd.DataFrame(linhas)
    cats = res["desfecho"].isin(CATEGORIAS) & res["p_cmh"].notna()
    res["p_cmh_holm"] = np.nan
    if cats.sum():
        res.loc[cats, "p_cmh_holm"] = multipletests(res.loc[cats, "p_cmh"], method="holm")[1]
    res["significativo_cmh"] = res["p_cmh"] < ALFA
    return res


TESTES = testar(PRES)
TESTES_CONS = testar(PRES_CONS)
if TESTES is None:
    print("Sem aplicações nos dois grupos: os testes serão calculados quando houver relatórios low-code.")
else:
    TABELAS["4.5a_testes_principal"] = TESTES.round(4)
    TABELAS["4.5b_testes_conservador"] = TESTES_CONS.round(4)
    print("CENÁRIO PRINCIPAL")
    display(TESTES.round(3))
    print("CENÁRIO CONSERVADOR")
    display(TESTES_CONS.round(3))

In [ ]:
# Figura 5: forest plot do OR de Mantel-Haenszel por categoria (escala log), um por cenário.
def forest(testes, cenario, arquivo):
    t = testes[testes["desfecho"].isin(CATEGORIAS)].copy()
    finitos = t[np.isfinite(t["or_mh"]) & (t["or_mh"] > 0) & t["or_mh_ic95_inf"].notna()]
    fig, ax = plt.subplots(figsize=(7, 4.8))
    y = np.arange(len(t))
    pos = dict(zip(t["desfecho"], y))
    for _, r in finitos.iterrows():
        cor = COR_GRUPO["vibe"] if r["hipotese"] == "H1" else TINTA_2
        ax.plot([r["or_mh_ic95_inf"], r["or_mh_ic95_sup"]], [pos[r["desfecho"]]] * 2, color=cor, lw=2)
        ax.scatter(r["or_mh"], pos[r["desfecho"]], s=48, color=cor, zorder=3, edgecolor="white")
    for _, r in t.drop(finitos.index).iterrows():
        if r["or_mh"] == 0:
            motivo = "OR = 0: achado só aparece em low-code"
        elif np.isinf(r["or_mh"]):
            motivo = "OR infinito: achado só aparece em vibe coding"
        else:
            motivo = "não estimável: mesma proporção (0% ou 100%) nos dois grupos"
        ax.text(0.02, pos[r["desfecho"]], motivo, transform=ax.get_yaxis_transform(),
                va="center", fontsize=8, color=TINTA_2, zorder=5,
                bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
    ax.axvline(1, color=TINTA_2, lw=1, ls="--", zorder=1)
    ax.set_xscale("log")
    if finitos.empty:
        ax.set_xlim(0.1, 10)
    ax.set_ylim(len(t) - 0.5, -0.5)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{d} {NOME_CAT[d]}" for d in t["desfecho"]])
    ax.set_xlabel("OR de Mantel-Haenszel (IC 95%), escala log. OR > 1: maior chance em vibe coding")
    ax.set_title(f"Vibe coding × low-code por domínio, {ROTULO_CENARIO[cenario].lower()} (azul = H1)")
    ax.grid(axis="y", visible=False)
    salvar_figura(fig, arquivo)
    plt.show()


if TESTES is not None:
    forest(TESTES, "principal", "fig5a_forest_or_mh_principal")
    forest(TESTES_CONS, "conservador", "fig5b_forest_or_mh_conservador")

## 4.6 Análise de sensibilidade

Comparação lado a lado dos dois cenários para os desfechos da H1 e os gerais:

* **principal:** alertas incluídos na rodada final (`--excluir-cabecalhos --excluir-pendentes`);
* **conservador:** só alertas validados como VP ou, quando a validação não é exigida, com confiança Média ou superior.

Se a conclusão da H1 se mantém nos dois cenários, o resultado é robusto ao critério de contagem; se não se mantém, ela depende dos alertas de confiança Baixa.

In [ ]:
sens = []
for nome, filtro in CENARIOS.items():
    pres_c, inc_c = presenca(filtro)
    r = testar(pres_c, ["A05", "A07", "qualquer_achado", "medio_ou_mais", "alto"])
    linha_base = {"cenario": nome, "achados_incluidos": len(inc_c),
                  "media_achados_por_app": pres_c["n_achados"].mean()}
    if r is None:
        sens.append(linha_base)
        continue
    for _, x in r.iterrows():
        sens.append({**linha_base, "desfecho": x["desfecho"], "pct_vibe": x["pct_vibe"],
                     "pct_lowcode": x["pct_lowcode"], "or_mh": x["or_mh"],
                     "or_mh_ic95_inf": x["or_mh_ic95_inf"], "or_mh_ic95_sup": x["or_mh_ic95_sup"],
                     "p_cmh": x["p_cmh"], "p_fisher": x["p_fisher"]})
SENS = pd.DataFrame(sens)
TABELAS["4.6_sensibilidade"] = SENS.round(4)
display(SENS.round(3))

## Hipótese H2 (opcional)

A H2 (segredos fixos ou previsíveis no código) não é medida pelo ZAP. Quando a busca de segredos estiver pronta, salve um CSV com as colunas `app` e `segredo` (0 ou 1, uma linha por aplicação, com o mesmo identificador `<grupo>__<fonte>__<dominio>`), aponte `ARQUIVO_SEGREDOS` para ele na configuração e rode a célula abaixo. O teste usa os mesmos métodos da seção 4.5.

In [ ]:
if ARQUIVO_SEGREDOS:
    seg = ler_csv(ARQUIVO_SEGREDOS)
    seg["segredo"] = pd.to_numeric(seg["segredo"], errors="coerce").fillna(0).astype(int)
    pres_h2 = apps.merge(seg[["app", "segredo"]], on="app", how="inner")
    print(f"Aplicações com avaliação de segredos: {len(pres_h2)} de {len(apps)}")
    H2 = testar(pres_h2, ["segredo"])
    if H2 is not None:
        H2["hipotese"] = "H2"
        TABELAS["h2_segredos"] = H2.round(4)
        display(H2.round(3))
else:
    print("ARQUIVO_SEGREDOS não definido: H2 não calculada.")

## Números-chave para o texto

Valores de referência para a redação do Capítulo 4, já formatados com vírgula decimal.

In [ ]:
print("AMOSTRA")
for g, n in PRES.groupby("grupo").size().items():
    print(f"  {ROTULO_GRUPO[g]}: {n} aplicações testadas")
print(f"  Total: {len(PRES)} aplicações testadas; {len(ausentes)} sem relatório por erro grave que impediu o teste (não funcionais)")

print("\nACHADOS POR APLICAÇÃO (cenário principal)")
for g, sub in PRES.groupby("grupo"):
    print(f"  {ROTULO_GRUPO[g]}: média {num(sub['n_achados'].mean(), 2)}, "
          f"mediana {num(float(sub['n_achados'].median()), 1)}, "
          f"desvio padrão {num(sub['n_achados'].std(), 2)}; "
          f"{num(100 * sub['alto'].mean())}% das apps com ao menos um achado Alto")

print("\nCATEGORIAS MAIS PREVALENTES (todas as aplicações)")
geral = PRES[CATEGORIAS].mean().mul(100).sort_values(ascending=False)
for cat, pct in geral.head(5).items():
    print(f"  {cat} {NOME_CAT[cat]}: {num(pct)}% das aplicações")

print("\nALERTAS MAIS FREQUENTES")
for _, r in freq.head(5).iterrows():
    print(f"  {r['alerta']} ({r['owasp_2025'] or 'sem categoria'}, risco {r['risco']}): "
          f"{r['n_apps']} apps ({num(r['pct_apps_total'])}%)")

if TESTES is None:
    print("\nHIPÓTESE H1: ainda não calculada (faltam aplicações low-code).")
else:
    for cen, tt in (("principal", TESTES), ("conservador", TESTES_CONS)):
        print(f"\nHIPÓTESE H1 ({ROTULO_CENARIO[cen].lower()})")
        for _, r in tt[tt["hipotese"] == "H1"].iterrows():
            if r["casos_vibe"] == 0 and r["casos_lowcode"] == 0:
                print(f"  {r['desfecho']} {r['categoria']}: nenhum achado nos dois grupos; "
                      "OR não estimável e hipótese não testável com o ZAP")
                continue
            print(f"  {r['desfecho']} {r['categoria']}: vibe {num(r['pct_vibe'])}% "
                  f"({int(r['casos_vibe'])}/{int(r['n_vibe'])}) × low-code {num(r['pct_lowcode'])}% "
                  f"({int(r['casos_lowcode'])}/{int(r['n_lowcode'])}); OR_MH = {num(r['or_mh'], 2)} "
                  f"(IC 95% {num(r['or_mh_ic95_inf'], 2)} a {num(r['or_mh_ic95_sup'], 2)}), "
                  f"p CMH = {num(r['p_cmh'], 4)}; p Fisher = {num(r['p_fisher'], 4)}")
if len(pendentes):
    print(f"\nLembrete: {len(pendentes)} alertas pendentes de validação; números provisórios.")

## Exportação

In [ ]:
caminho_xlsx = os.path.join(PASTA_SAIDA, "tabelas_cap4.xlsx")
with pd.ExcelWriter(caminho_xlsx) as w:
    for nome, df in TABELAS.items():
        df.to_excel(w, sheet_name=nome[:31], index=False)
        df.to_csv(os.path.join(PASTA_SAIDA, nome + ".csv"), sep=";", decimal=",",
                  index=False, encoding="utf-8-sig")
print("Tabelas salvas em", caminho_xlsx)

if NO_COLAB:
    from google.colab import files
    arquivo_zip = PASTA_SAIDA.rstrip("/") + ".zip"
    with zipfile.ZipFile(arquivo_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for raiz, _, nomes in os.walk(PASTA_SAIDA):
            for nome in nomes:
                z.write(os.path.join(raiz, nome))
    files.download(arquivo_zip)